# Практическое занятие 07. Люди на фото и видео


In [ ]:
!pip install numpy opencv-contrib-python mediapipe onnxruntime matplotlib pandas imageio-ffmpeg

### Подготовка


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT.resolve()))
import lab_utils as lab
import practice_engine as engine

PHOTOS, VIDEOS, RESULTS = ROOT/'data/photos', ROOT/'data/videos', ROOT/'results'
(RESULTS/'videos').mkdir(parents=True, exist_ok=True)
cv2.setNumThreads(2)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)
plt.rcParams.update({'figure.figsize': (11, 4), 'figure.dpi': 110})
print('Фото LFW:', len(list((PHOTOS/'lfw').glob('*/*.jpg'))))
print('Данные и модели читаются локально. Подготовка завершена.')

---
## 1. Поза человека: что модель возвращает и зачем

Поза описывает человека через суставы. Это компактное промежуточное представление: нейросеть извлекает точки из пикселей, а дальше мы можем обычной геометрией измерять угол колена, положение рук или движение таза.

**MediaPipe Pose / GHUM** использует каскад поиска тела и оценки ориентиров. В нём 33 точки. **RTMPose-t** в нашем экспорте получает кроп человека и выдаёт 17 суставов COCO; его backbone CSPNeXt извлекает признаки, а голова SimCC предсказывает положение по двум осям. Поиск человека перед RTMPose здесь заменён известной областью из разметки.

Начнём с реальных ситуаций: йога, лыжи, сидящий человек, разворот бейсболиста, присевший игрок и упражнение. COCO-разметка нужна только для выбора кропа человека; сравнение времени не включает этот внешний детектор.

In [ ]:
pose_cases = engine.pose_cases()
lab.show_grid([case['image'] for case in pose_cases],
              [case['name'] for case in pose_cases], cols=4, size=3.2)
print('Реальных примеров:', len(pose_cases))

### Одни и те же люди — две схемы скелета

Выполним обе модели на каждом изображении. RTMPose получает весь показанный кроп, MediaPipe ищет ведущую позу внутри него. Рисуем только достаточно уверенные точки. Отсутствие скелета — результат пропуска, его не заменяем нулевыми координатами.

In [ ]:
lab.diagram('Две архитектуры оценки позы', [
    ['Кадр', 'Детектор тела → ROI', 'MediaPipe / GHUM', '33 координаты и visibility'],
    ['Кроп человека', 'CSPNeXt: признаки', 'SimCC: отдельные оси X/Y', '17 суставов COCO'],
])
pose_model = lab.make_pose()
pose_predictions = []
pose_comparison = []
for case in pose_cases:
    image = case['image']
    start = time.perf_counter()
    mp_points = lab.pose_points(pose_model.detect(lab.mp_image(image)), image.shape)
    mp_ms = (time.perf_counter()-start)*1000
    start = time.perf_counter()
    roi = None
    if case['source']=='pose.jpg':
        H,W = image.shape[:2]; roi = [.20*W,.28*H,.60*W,.68*H]
    rt_points, debug = engine.rtmpose_debug(image,roi)
    rt_ms = (time.perf_counter()-start)*1000
    pose_predictions.append({'mp': mp_points, 'rt': rt_points, 'debug': debug})
    pose_comparison.append({'пример': case['name'], 'MediaPipe найдено': mp_points is not None,
                            'MP мс': mp_ms, 'RTM мс': rt_ms})
    lab.show_grid([lab.draw_pose(image, mp_points), lab.draw_pose(image, rt_points, lab.COCO_EDGES)],
                  [case['name']+' | MediaPipe', case['name']+' | RTMPose'], cols=2, size=3.7)
display(pd.DataFrame(pose_comparison).round(1))


### Координаты — на том самом человеке, которого мы видим

Чтобы нарисовать сустав, нужна его позиция в исходном изображении. MediaPipe возвращает нормированные `x` и `y`; например, `x=0.5` означает середину ширины. Перевод выполняется как `x_px=x*W`, `y_px=y*H`.

Покажем **реальное предсказание запястья** на фото с йогой, его координаты и увеличенный участок. Важно, что `image.shape` имеет порядок `(H, W, 3)`, а точка задаётся как `(x, y)`.

In [ ]:
image = pose_cases[0]['image']
result = pose_model.detect(lab.mp_image(image))
point = result.pose_landmarks[0][15]  # левое запястье по схеме MediaPipe
height, width = image.shape[:2]
x_px, y_px = point.x*width, point.y*height
overlay = image.copy()
cv2.circle(overlay, (round(x_px), round(y_px)), 10, (0, 0, 255), 3)
cv2.putText(overlay, 'wrist 15', (round(x_px)+10, round(y_px)-10),
            cv2.FONT_HERSHEY_SIMPLEX, .6, (0, 0, 255), 2)
zoom, _ = engine.crop_box(overlay, [x_px-45, y_px-45, 90, 90], padding=0)
lab.show_grid([overlay, zoom], ['Запястье в исходном кадре', 'Увеличенный реальный фрагмент'], cols=2, size=4)
display(pd.DataFrame([{'W': width, 'H': height, 'x_normalized': point.x,
                      'y_normalized': point.y, 'x_px': x_px, 'y_px': y_px}]))

### Почему точек 33 и 17 — и что добавляет MediaPipe

У обеих схем есть плечи, локти, запястья, таз, колени и лодыжки. MediaPipe добавляет ориентиры лица, по несколько точек на кистях и ориентиры пятки/носка

In [ ]:
p = pose_predictions[0]['mp']
hand_indices = [15, 17, 19, 21]
foot_indices = [27, 29, 31]
extra_points = []
for indices in [hand_indices, foot_indices]:
    selected = p[indices, :2]
    out = image.copy()
    for index in indices:
        x, y = p[index, :2].astype(int)
        cv2.circle(out, (x, y), 5, (0, 0, 255), -1)
        cv2.putText(out, str(index), (x+5, y-5), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 0, 255), 1)
    minimum, maximum = selected.min(0)-25, selected.max(0)+25
    crop, _ = engine.crop_box(out, [*minimum, *(maximum-minimum)], padding=0)
    extra_points.append(crop)
lab.show_grid(extra_points, ['Кисть: 15, 17, 19, 21', 'Стопа: 27, 29, 31'], cols=2, size=4)

### Как RTMPose получает координату: реальный выход сети

Вернёмся к тому же запястью. В нашей RTMPose это индекс 9. Сеть выдаёт не нарисованную нами карту, а два массива: оценки X и Y. На каждом выбираем максимальное значение. В этом экспорте два бина соответствуют одному пикселю кропа: делим индекс на 2. Затем обратная affine-матрица возвращает точку на исходное фото.


In [ ]:
debug = pose_predictions[0]['debug']
joint = 9
local_x, local_y = debug['local'][joint]
crop = debug['crop'].copy()
cv2.circle(crop, (round(float(local_x)), round(float(local_y))), 5, (0, 0, 255), -1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
axes[0].imshow(crop[:, :, ::-1]); axes[0].set_title('Вход RTMPose: 192×256')
axes[1].plot(np.arange(len(debug['x'][joint]))/2, debug['x'][joint])
axes[1].axvline(local_x, color='r'); axes[1].set(title='Реальные оценки X', xlabel='x в кропе, px')
axes[2].plot(np.arange(len(debug['y'][joint]))/2, debug['y'][joint])
axes[2].axvline(local_y, color='r'); axes[2].set(title='Реальные оценки Y', xlabel='y в кропе, px')
plt.tight_layout(); plt.show()
print('Точка в кропе:', debug['local'][joint].round(1))
print('Та же точка в исходном фото:', pose_predictions[0]['rt'][joint, :2].round(1))

---
## 2. Лицо: зачем бокс, ориентиры и выравнивание

Наша конечная задача — сравнивать лица между разными фотографиями и кадрами. Для этого сначала нужно получить **правильный вход модели**: найти лицо, убедиться, что в кропе достаточно деталей, и расположить его предсказуемым образом.

Haar cascade проверяет простые признаки яркости. YuNet извлекает свёрточные признаки и возвращает бокс вместе с пятью ориентирами. MediaPipe Face Landmarker уточняет сотни ориентиров. Эти инструменты решают разные части подготовки входа.

Посмотрим на реальные повороты и наклоны головы, а не только на искусственно затемнённую картинку. В непрерывном видео люди могут сохранять личность, но сильно менять видимость деталей лица.

In [ ]:
face_cases = engine.face_cases()
haar = cv2.CascadeClassifier(str(ROOT/'models/haarcascade_frontalface_default.xml'))
face_detection_rows = []
for case in face_cases:
    image = case['image']
    old = haar.detectMultiScale(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), 1.1, 5)
    new = lab.detect_faces(image)
    face_detection_rows.append({'пример': case['name'], 'Haar': len(old), 'YuNet': len(new)})
    lab.show_grid([lab.draw_faces(image, old), lab.draw_faces(image, new)],
                  [case['name']+' | Haar', case['name']+' | YuNet'], cols=2, size=3.5)
display(pd.DataFrame(face_detection_rows))

### Что именно содержится в ориентире

Строка YuNet содержит `[x, y, width, height]`, пять пар `(x, y)` и score детектора. Для рисования правого нижнего угла вычисляем `(x+width, y+height)`; для кропа массива — `image[y:y+h, x:x+w]`.

Пять точек позволяют оценить преобразование подобия: поворот, общий масштаб и сдвиг. После выравнивания глаза и рот оказываются около стандартных мест. При сильном ракурсе или неверных ориентирах преобразование может ухудшить вход — это увидим на реальных примерах.

In [ ]:
alignment_images, alignment_titles = [], []
for case in face_cases:
    image = case['image']; face = lab.largest_face(image)
    if face is None: continue
    marked = lab.draw_faces(image, [face])
    raw, _ = engine.crop_box(marked, face[:4], padding=.15)
    aligned = lab.align_face(image, face)
    alignment_images.extend([raw, aligned])
    alignment_titles.extend([case['name']+' | бокс + 5 точек', 'Выровненный вход 112×112'])
lab.show_grid(alignment_images, alignment_titles, cols=4, size=2.8)


### Face landmarks крупным планом

Плотные ориентиры нужны, когда пяти точек мало: например, для расположения границ рта или глаз. Здесь разберём их на нескольких реальных ракурсах. Плотная сетка показывает форму, но сама по себе не является идентификатором человека.

Для понятности рисуем полный результат и отдельно увеличиваем область глаз и губ. Ниже используем двумерные координаты; не вводим отдельный 3D-анализ головы.

In [ ]:
import mediapipe as mp
mesh_model = mp.tasks.vision.FaceLandmarker.create_from_options(mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(ROOT/'models/face.task')), num_faces=1))
mesh_examples = []
for case in face_cases[:4] + face_cases[4:6]:
    source = case['image']; face = lab.largest_face(source)
    if face is None: continue
    crop, _ = engine.crop_box(source, face[:4], padding=.3)
    points = engine.mesh_points(mesh_model, crop)
    if points is None: continue
    full = crop.copy()
    for x, y in points: cv2.circle(full, (int(x), int(y)), 1, (0, 255, 0), -1)
    eyes = engine.marked_region(crop, points, [33, 133, 160, 144, 263, 362, 385, 380])
    mouth = engine.marked_region(crop, points, [61, 291, 13, 14, 78, 308, 0, 17])
    lab.show_grid([crop, full, eyes, mouth],
                  [case['name'], f'Полная сетка: {len(points)}', 'Глаза крупно', 'Губы крупно'], cols=4, size=2.7)
mesh_model.close()


---
## 3. Видео
### 3.1. Считаем приседания


Возьмём таз, колено и голеностоп одной стороны. Угол в колене — угол между векторами `таз−колено` и `голеностоп−колено`. Ноги в кадре выбираем по visibility. При слабой уверенности измерение будет отсутствующим, а не равным нулю.

Дополнительно измерим высоту таза над лодыжкой, делённую на длину торса. Это второе простое правило: при приседании значение обычно уменьшается. Нормировка помогает при изменении масштаба, но не устраняет зависимость от ракурса.

In [ ]:
body_frames, body_times, body_fps, body_points, measurements = engine.analyze_squats()
squat_table = pd.DataFrame(measurements)
display(squat_table[['time_s', 'knee_angle', 'hip_height', 'visibility']].head())

# Покажем настоящие опорные точки и геометрию одного стоящего и одного присевшего кадра.
examples = []
for target in [0, 1.9]:
    i = int(np.argmin(np.abs(body_times-target)))
    out = lab.draw_pose(body_frames[i], body_points[i])
    examples.append(out)
lab.show_grid(examples, ['Стоит: угол близок к развёрнутому', 'Присел: угол уменьшился'], cols=2, size=4)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(squat_table.time_s, squat_table.knee_angle); axes[0].set(xlabel='секунды', ylabel='угол колена, °')
axes[1].plot(squat_table.time_s, squat_table.hip_height); axes[1].set(xlabel='секунды', ylabel='высота таза / длина торса')
plt.tight_layout(); plt.show()

### Почему одного порога недостаточно

Если считать повтор на каждом кадре с углом <105°, одно приседание превратится в десятки повторений. Дрожание около одного порога тоже создаёт ложные переходы.

Используем два порога и состояние: **стоим → опускаемся → внизу → поднимаемся → снова стоим**. Полный повтор засчитываем только после возврата в положение стоя. Угол слегка сгладим EMA; пропуск >0.5 секунды сбрасывает незавершённый цикл, чтобы не склеить разные движения.


In [ ]:
class SquatCounter:
    """Считает завершённый цикл standing → bottom → standing."""
    def __init__(self, down=105, up=160):
        self.down, self.up = down, up
        self.armed = False; self.bottom_seen = False; self.count = 0
        self.filtered = None; self.last_good = None
    def update(self, angle, t):
        if not np.isfinite(angle):
            return 'unknown', self.count, np.nan
        if self.last_good is not None and t-self.last_good > .5:
            self.armed = self.bottom_seen = False; self.filtered = None
        self.last_good = t
        self.filtered = angle if self.filtered is None else .55*angle + .45*self.filtered
        value = self.filtered
        if value >= self.up:
            if self.armed and self.bottom_seen: self.count += 1
            self.armed = True; self.bottom_seen = False
            phase = 'standing'
        elif value <= self.down:
            if self.armed: self.bottom_seen = True
            phase = 'bottom'
        else:
            phase = 'rising' if self.bottom_seen else 'lowering'
        return phase, self.count, value

counter = SquatCounter()
states = [counter.update(row.knee_angle, row.time_s) for row in squat_table.itertuples()]
squat_table[['state', 'count', 'filtered_angle']] = pd.DataFrame(states, index=squat_table.index)
squat_table['naive_low_frame'] = squat_table.knee_angle < 105
# Второе правило по относительной высоте таза: пороги также только для этого демо.
HIP_BOTTOM, HIP_STANDING = 1.10, 1.50
squat_table['hip_rule'] = np.select(
    [squat_table.hip_height<=HIP_BOTTOM, squat_table.hip_height>=HIP_STANDING],
    ['bottom', 'standing'], default='transition')
squat_table.loc[squat_table.hip_height.isna(),'hip_rule'] = 'unknown'
print('Кадров ниже порога:', int(squat_table.naive_low_frame.sum()))
print('Завершённых повторений:', int(squat_table['count'].iloc[-1]))
display(pd.crosstab(squat_table.state, squat_table.hip_rule))
squat_table.to_csv(RESULTS/'squat_states.csv', index=False)

In [ ]:
state_colors = {'standing': (0,180,0), 'lowering': (0,150,255), 'bottom': (0,0,220),
                'rising': (220,100,0), 'unknown': (130,130,130)}
squat_output = []
for frame, points, row in zip(body_frames, body_points, squat_table.itertuples()):
    out = lab.draw_pose(frame, points)
    cv2.putText(out, f'{row.state} | reps: {int(row.count)}', (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, .52, state_colors[row.state], 2)
    if np.isfinite(row.filtered_angle):
        cv2.putText(out, f'knee: {row.filtered_angle:.0f} deg', (8, 44),
                    cv2.FONT_HERSHEY_SIMPLEX, .48, (0,0,0), 1)
    squat_output.append(out)
lab.save_video(squat_output, RESULTS/'videos/squat_counter.mp4', body_fps)
fig, ax = plt.subplots()
ax.plot(squat_table.time_s, squat_table.knee_angle, alpha=.4, label='исходный угол')
ax.plot(squat_table.time_s, squat_table.filtered_angle, label='EMA')
ax.axhline(105, color='r', ls='--', label='внизу'); ax.axhline(160, color='g', ls='--', label='стоит')
for _, phase in squat_table.groupby((squat_table.state!=squat_table.state.shift()).cumsum()):
    color = np.array(state_colors[phase.state.iloc[0]])[::-1]/255
    ax.axvspan(phase.time_s.iloc[0],phase.time_s.iloc[-1],color=color,alpha=.08)
events = squat_table['count'].diff().fillna(0)>0
for t in squat_table.loc[events,'time_s']: ax.axvline(t, color='k', alpha=.4)
ax.set(xlabel='секунды', ylabel='угол колена, °', title='Вертикальные линии — завершённые повторения')
ax.legend(); plt.show()



### 3.2. Устойчивые треки лиц без распознавания личности

Перейдём к непрерывному ролику с двумя людьми. Они наклоняют и поворачивают голову, а камера не делает частых склеек.


In [ ]:
main_video = VIDEOS/'head-pose-face-detection-female-and-male.mp4'
face_analysis = engine.analyze_faces(main_video, start=0, duration=24, sample_fps=6)
print('ID простого IoU-трекера:', face_analysis['baseline_tracks'])
print('ID с движением и удержанием:', face_analysis['motion_tracks'])
lab.save_video(engine.draw_tracks(face_analysis), RESULTS/'videos/stable_face_tracks.mp4', face_analysis['fps'])

fig, ax = plt.subplots(figsize=(11, 3))
for tid in sorted(set(r['track'] for r in face_analysis['records'])):
    ts = [r['time'] for r in face_analysis['records'] if r['track']==tid]
    ax.scatter(ts, [tid]*len(ts), s=9, label=f'track {tid}')
ax.set(xlabel='секунды исходного ролика', ylabel='track ID', title='Покрытие геометрических треков')
ax.legend(); plt.show()

---
## 4. Эмбеддинги: сравниваем реальные снимки одного человека

После выравнивания **MobileFaceNet** превращает лицо в вектор. В пакете InsightFace с ArcFace получаем 512 координат; SFace из OpenCV возвращает 128. ArcFace и SFace — разные способы обучения; пространства двух моделей не смешиваем.

Близость нормированных векторов вычисляем как $s=a^T b$. Посмотрим на настоящие пары в разных ракурсах и матрицу сходства.

In [ ]:
lab.diagram('Зачем landmarks нужны распознаванию', [
    ['BGR-фото', 'YuNet: лицо и 5 точек', 'Выравнивание 112×112', 'Сеть → нормированный вектор'],
    ['ArcFace: обучение', 'Снимки одного человека ближе', 'Другие люди дальше', 'При поиске: скалярное произведение'],
])
# Эти референсы берём из более поздних кадров, не попадающих в основной фрагмент 0–24 с.
# A — человек слева, B — справа; это локальные анонимные метки учебных данных.
enrollment = engine.make_enrollment()
lab.show_grid([r['image'] for r in enrollment],
              [r['person']+' | '+r['file'] for r in enrollment], cols=3, size=3)
print('Входные референсы сохранены в data/photos/enrolled/.')

### Одно лицо в разных ракурсах и разные лица

Сравним референс A с тем же человеком на обычном кадре и при сильном наклоне, затем с B. Не предполагаем, что все ракурсы должны успешно распознаться: на предыдущем шаге видели, как меняются ориентиры и кроп.

На матрице ниже сначала идут разные изображения A, затем B. Сравниваем **рисунок матрицы**: близки ли снимки одной группы и отделяются ли группы? Шкалы ArcFace и SFace нельзя читать как одинаково откалиброванную уверенность.

In [ ]:
pair_images = [enrollment[0]['image'], enrollment[1]['image']]
pair_titles = ['A: референс 1', 'A: референс 2']
for seconds in [0, 10]:
    frame = engine.frame_at(main_video, seconds)
    faces = lab.detect_faces(frame)
    left = min(faces, key=lambda f:f[0]+f[2]/2)
    crop, _ = engine.crop_box(frame, left[:4], padding=.3)
    pair_images.append(crop); pair_titles.append(f'A: другой ракурс {seconds} с')
pair_images.extend([enrollment[3]['image'], enrollment[4]['image']])
pair_titles.extend(['B: референс 1', 'B: референс 2'])
lab.show_grid(pair_images, pair_titles, cols=3, size=3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
pair_vectors = {}
for ax, model_name in zip(axes, ['ArcFace', 'SFace']):
    vectors = np.stack([lab.encode_face(im, model=model_name) for im in pair_images])
    pair_vectors[model_name] = vectors
    matrix = vectors @ vectors.T
    im = ax.imshow(matrix, vmin=-.1, vmax=1, cmap='viridis')
    for i in range(len(matrix)):
        for j in range(len(matrix)): ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_xticks(range(6), ['A1','A2','A0s','A10s','B1','B2'])
    ax.set_yticks(range(6), ['A1','A2','A0s','A10s','B1','B2'])
    ax.set_title(model_name)
    fig.colorbar(im, ax=ax, fraction=.045)
plt.tight_layout(); plt.show()

### Отдельно сравним «вход сети», а не только исходные фотографии

Сильный наклон может дать уверенный бокс, но неточные пять точек. Поэтому низкое сходство иногда начинается с неверной геометрии входа. Увеличим именно выровненные кропы.

In [ ]:
aligned_pairs = [lab.align_face(im, lab.largest_face(im)) for im in pair_images]
lab.show_grid(aligned_pairs, pair_titles, cols=3, size=2.8)
for model_name, vectors in pair_vectors.items():
    print(model_name, 'A1↔A2:', round(float(vectors[0]@vectors[1]),3),
          '| A1↔A10s:', round(float(vectors[0]@vectors[3]),3),
          '| A1↔B1:', round(float(vectors[0]@vectors[4]),3))

---
## 5. Поиск query по датасету фотографий

Теперь собираем обычный индекс лиц. В LFW сохранены **30 людей × 8 разных фото**. По два снимка каждого идут в галерею, остальные — query. Ищем другой снимок человека, а не копию картинки. Поиск — матрица эмбеддингов галереи, скалярное произведение и сортировка.


In [ ]:
photo_records = []
photo_vectors = {'ArcFace': [], 'SFace': []}
for person_dir in sorted((PHOTOS/'lfw').iterdir()):
    for photo_number, path in enumerate(sorted(person_dir.glob('*.jpg'))):
        image = cv2.imread(str(path)); face = lab.largest_face(image)
        if face is None: continue
        photo_records.append({'path':path, 'person':person_dir.name, 'photo':photo_number,
                              'image':image, 'crop':lab.align_face(image,face)})
        for model_name in photo_vectors:
            photo_vectors[model_name].append(lab.encode_face(image,face,model=model_name))
for model_name in photo_vectors: photo_vectors[model_name] = np.stack(photo_vectors[model_name])
gallery = np.array([i for i,r in enumerate(photo_records) if r['photo']<2])
queries = np.array([i for i,r in enumerate(photo_records) if r['photo']>=2])
assert not set(gallery)&set(queries)
print('Успешно обработано фото:',len(photo_records),'| галерея:',len(gallery),'| query:',len(queries))
print('Форма ArcFace gallery:',photo_vectors['ArcFace'][gallery].shape)

def photo_search(query_number, model_name='ArcFace', query_vector=None, title=None):
    """Одинаковая выдача: query, три кандидата, score и исходный файл."""
    if query_vector is None: query_vector = photo_vectors[model_name][query_number]
    scores = photo_vectors[model_name][gallery] @ query_vector
    order = np.argsort(-scores)[:3]
    hits = [{'image':photo_records[gallery[j]]['crop'], 'label':photo_records[gallery[j]]['person'],
             'score':float(scores[j]), 'file':photo_records[gallery[j]]['path'].name, 'time_s':None} for j in order]
    return lab.show_search_results(photo_records[query_number]['crop'],hits,
                                  title or f'{model_name}: {photo_records[query_number]["path"].name}')


Для каждого query посчитаем место первого снимка правильного человека, лучший score правильного и лучшего другого человека.

In [ ]:
evaluation_rows = []
for model_name, vectors in photo_vectors.items():
    scores = vectors[queries] @ vectors[gallery].T
    for i, query_number in enumerate(queries):
        correct = np.array([photo_records[j]['person']==photo_records[query_number]['person'] for j in gallery])
        order = np.argsort(-scores[i])
        rank = int(np.flatnonzero(correct[order])[0])+1
        same = float(scores[i,correct].max()); different = float(scores[i,~correct].max())
        evaluation_rows.append({'model':model_name,'query':query_number,'correct_rank':rank,
                                'same_score':same,'other_score':different,'gap':same-different})
photo_eval = pd.DataFrame(evaluation_rows)
display(photo_eval.groupby('model').agg(query_count=('query','size'),
        correct_top1=('correct_rank',lambda r:int((r==1).sum())),mean_rank=('correct_rank','mean')))
arc_eval = photo_eval[photo_eval.model=='ArcFace']
easy_query = int(arc_eval.sort_values('gap',ascending=False).iloc[0]['query'])
hard_query = int(arc_eval.sort_values('gap').iloc[0]['query'])
mid_query = int(arc_eval.sort_values('same_score').iloc[len(arc_eval)//2]['query'])
for label, query_number in [('Большой запас',easy_query),('Самый трудный по запасу',hard_query),('Обычный запрос',mid_query)]:
    photo_search(query_number,'ArcFace',title=label+' | ArcFace')

### ArcFace и SFace

Покажем обе модели на одном сложном запросе и проверим, где они расходятся по первому кандидату.

In [ ]:
photo_search(hard_query,'SFace',title='Тот же трудный query | SFace')
disagreements = []
for query_number in queries:
    winners = []
    for model_name in ['ArcFace','SFace']:
        sims = photo_vectors[model_name][gallery] @ photo_vectors[model_name][query_number]
        winners.append(photo_records[gallery[int(sims.argmax())]]['person'])
    if winners[0]!=winners[1]: disagreements.append(int(query_number))
print('Расхождений по человеку top-1:',len(disagreements))
if disagreements:
    for model_name in ['ArcFace','SFace']: photo_search(disagreements[0],model_name)
display(photo_eval[photo_eval['query']==hard_query])

### Разбор ошибок

Проведём один и тот же контролируемый опыт на **трёх разных людях**. Меняем только query, галерея остаётся прежней.

In [ ]:
challenging_queries = []
seen_people = set()
for row in arc_eval.sort_values('gap').itertuples():
    if photo_records[row.query]['person'] not in seen_people:
        challenging_queries.append(int(row.query)); seen_people.add(photo_records[row.query]['person'])
    if len(challenging_queries)==3: break

degradation_rows = []
degradation_details = []
variant_dir = RESULTS/'query_variants'
variant_dir.mkdir(exist_ok=True)
for query_number in challenging_queries:
    record = photo_records[query_number]; image = record['image']; face = lab.largest_face(image)
    x,y,w,h = face[:4].astype(int)
    raw = image[max(0,y):y+h,max(0,x):x+w]
    variants = {'original':record['crop'], 'blur_6':cv2.GaussianBlur(record['crop'],(0,0),6),
                'size_8':cv2.resize(cv2.resize(record['crop'],(8,8)),(112,112)),
                'no_alignment':cv2.resize(raw,(112,112))}
    lab.show_grid(list(variants.values()),[record['person']+' | '+k for k in variants],cols=4,size=2.7)
    for condition,crop in variants.items():
        cv2.imwrite(str(variant_dir/f'{record["person"]}_{condition}.jpg'),crop)
        vector = lab.arcface_crop(crop); sims = photo_vectors['ArcFace'][gallery]@vector
        order = np.argsort(-sims)
        same_mask = np.array([photo_records[j]['person']==record['person'] for j in gallery])
        rank = int(np.flatnonzero(same_mask[order])[0])+1
        correct_score = float(sims[same_mask].max()); other_score = float(sims[~same_mask].max())
        degradation_rows.append({'person':record['person'],'condition':condition,'correct_rank':rank,
                                 'correct_score':correct_score,'other_score':other_score,'gap':correct_score-other_score})
        degradation_details.append({'query':query_number,'condition':condition,'crop':crop,'sims':sims,'rank':rank})
degradation_table = pd.DataFrame(degradation_rows)
display(degradation_table.round(3))
degradation_table.to_csv(RESULTS/'photo_degradations.csv',index=False)
errors = [r for r in degradation_details if r['rank']>1]
for error in errors[:3]:
    sims = error['sims']; order = np.argsort(-sims)[:3]
    hits = [{'image':photo_records[gallery[j]]['crop'],'label':photo_records[gallery[j]]['person'],
             'score':float(sims[j]),'file':photo_records[gallery[j]]['path'].name,'time_s':None} for j in order]
    expected = photo_records[error['query']]['person']
    lab.show_search_results(error['crop'],hits,f'{error["condition"]}: ожидали {expected}')
print('Показаны фактические ошибки:',min(3,len(errors)),'из',len(errors),'ошибочных условий.')


---
## 6. Сохранённые люди и поиск по видео

Теперь разделим два состояния системы:

* `track_id`: геометрическая связь наблюдений, которую уже построили;
* `person_id`: запись сохранённого человека с референсными фотографиями.

У нас две анонимные записи — `person_A` и `person_B`, по три входных фото каждой.


In [ ]:
engine.add_embeddings(face_analysis)
tracks_all = engine.group_tracks(face_analysis['records'])
tracks_best = engine.group_tracks(face_analysis['records'],top_k=5)
person_ids = sorted(set(r['person'] for r in enrollment))
registry_vectors = np.stack([lab.unit(np.mean([r['vector'] for r in enrollment if r['person']==pid],axis=0))
                             for pid in person_ids])
quality_comparison = []
for all_track,best_track in zip(tracks_all,tracks_best):
    quality_comparison.append({'track':best_track['track'],
        'all→A':float(all_track['vector']@registry_vectors[0]),'all→B':float(all_track['vector']@registry_vectors[1]),
        'selected→A':float(best_track['vector']@registry_vectors[0]),'selected→B':float(best_track['vector']@registry_vectors[1])})
    lab.show_grid([r['crop'] for r in best_track['selected']],
                  [f'track {best_track["track"]} | {r["time"]:.2f}s' for r in best_track['selected']],cols=5,size=2.2)
display(pd.DataFrame(quality_comparison).round(3))


Для каждого трека сравниваем его вектор с референсами A и B.


In [ ]:
MATCH_THRESHOLD = .40
MIN_MARGIN = .10
person_map = {}
identity_rows = []
for track in tracks_best:
    scores = registry_vectors @ track['vector']
    order = np.argsort(-scores)
    margin = float(scores[order[0]]-scores[order[1]])
    accepted = scores[order[0]]>=MATCH_THRESHOLD and margin>=MIN_MARGIN
    identity = person_ids[order[0]] if accepted else 'unknown'
    person_map[track['track']] = identity
    identity_rows.append({'track':track['track'],'person_id':identity,'best_score':float(scores[order[0]]),
                          'second_score':float(scores[order[1]]),'margin':margin,'accepted':accepted})
    hits = []
    for j in order:
        reference = next(r for r in enrollment if r['person']==person_ids[j])
        hits.append({'image':lab.align_face(reference['image'],reference['face']), 'label':person_ids[j],
                     'score':float(scores[j]),'file':reference['file'],'time_s':None})
    lab.show_search_results(track['representative']['crop'],hits,
                            f'Track {track["track"]} → {identity}; отрыв {margin:.3f}')
identity_table = pd.DataFrame(identity_rows)
display(identity_table.round(3))
identity_table.to_csv(RESULTS/'track_to_person.csv',index=False)
lab.save_video(engine.draw_tracks(face_analysis,person_map),RESULTS/'videos/person_ids.mp4',face_analysis['fps'])

### Карта появлений и отказ, когда записи нет в галерее

Стабильный person ID относится к записи человека, а не к номеру одного трека. Покажем наблюдаемые появления A/B по времени.

In [ ]:
appearance_rows = []
fig, ax = plt.subplots(figsize=(11, 3))
for y,pid in enumerate(person_ids):
    observations = [r for r in face_analysis['records'] if person_map.get(r['track'])==pid]
    for start,end in engine.intervals([r['time'] for r in observations],max_gap=.5):
        appearance_rows.append({'person_id':pid,'file':main_video.name,'start_s':start,'end_s':end})
        ax.plot([start,end],[y,y],lw=8)
ax.set_yticks(range(len(person_ids)),person_ids)
ax.set(xlabel='секунды исходного ролика',title='Наблюдаемые появления сохранённых людей')
plt.tight_layout();plt.show()
appearance_table = pd.DataFrame(appearance_rows)
display(appearance_table.round(2));appearance_table.to_csv(RESULTS/'person_appearances.csv',index=False)

b_track = next(t for t in tracks_best if person_map[t['track']]=='person_B')
a_index = person_ids.index('person_A')
only_a_score = float(b_track['vector']@registry_vectors[a_index])
a_ref = next(r for r in enrollment if r['person']=='person_A')
lab.show_search_results(b_track['representative']['crop'],[
    {'image':lab.align_face(a_ref['image'],a_ref['face']),'label':'person_A — единственная запись',
     'score':only_a_score,'file':a_ref['file'],'time_s':None}], 'B отсутствует в галерее')
print('Решение:', 'person_A' if only_a_score>=MATCH_THRESHOLD else 'unknown',
      '| score:',round(only_a_score,3),'| учебный порог:',MATCH_THRESHOLD)

### Новый ролик и новые Track ID

В коридоре люди уходят и подходят снова, появляются и другие лица. Запустим **новый трекер** с отдельным диапазоном ID от 100.

In [ ]:
walking_analysis = engine.analyze_faces(VIDEOS/'face-demographics-walking-and-pause.mp4',
                                        duration=None,sample_fps=4,embeddings=True,track_start=100)
walking_tracks = engine.group_tracks(walking_analysis['records'],top_k=5)
walking_person_map = {}
walking_identity_rows = []
for track in walking_tracks:
    scores = registry_vectors@track['vector']; order = np.argsort(-scores)
    margin = float(scores[order[0]]-scores[order[1]])
    accepted = scores[order[0]]>=MATCH_THRESHOLD and margin>=MIN_MARGIN
    identity = person_ids[order[0]] if accepted else 'unknown'
    walking_person_map[track['track']] = identity
    walking_identity_rows.append({'file':'walking-and-pause.mp4','new_track_id':track['track'],
                                  'person_id':identity,'best_score':float(scores[order[0]]),'margin':margin})
walking_identity_table = pd.DataFrame(walking_identity_rows)
display(walking_identity_table.round(3))
walking_identity_table.to_csv(RESULTS/'walking_people.csv',index=False)
lab.save_video(engine.draw_tracks(walking_analysis,walking_person_map),
               RESULTS/'videos/people_reappear.mp4',walking_analysis['fps'])

# Покажем реально найденные повторные появления рядом с их референсом.
for pid in person_ids:
    candidates = [t for t in walking_tracks if walking_person_map[t['track']]==pid]
    if not candidates: continue
    track = max(candidates,key=lambda t:float(t['vector']@registry_vectors[person_ids.index(pid)]))
    ref = next(r for r in enrollment if r['person']==pid)
    lab.show_grid([lab.align_face(ref['image'],ref['face']),track['representative']['crop']],
                  [pid+' | сохранённый вход',f'{pid} | новый track {track["track"]}'],cols=2,size=3)

### Поиск по фотографии и по папке видео


In [ ]:
hamilton_analysis = engine.analyze_faces(VIDEOS/'hamilton.mp4',duration=None,sample_fps=6,embeddings=True)
hamilton_tracks = engine.group_tracks(hamilton_analysis['records'],top_k=5)
portrait = cv2.imread(str(PHOTOS/'queries/lin-manuel-miranda.png'))
portrait_crop = lab.align_face(portrait,lab.largest_face(portrait))
portrait_vector = lab.encode_face(portrait)
ranked_tracks = sorted(hamilton_tracks,key=lambda t:float(t['vector']@portrait_vector),reverse=True)
hits = []
for track in ranked_tracks[:3]:
    best_frame = max(track['records'],key=lambda r:float(r['vector']@portrait_vector))
    hits.append({'image':best_frame['crop'],'label':f'track {track["track"]}',
                 'score':float(track['vector']@portrait_vector),'file':'hamilton.mp4','time_s':best_frame['time']})
hamilton_result = lab.show_search_results(portrait_crop,hits,'Отдельный портрет → фрагменты Hamilton')
# Второе направление отказа: теперь query отсутствует в самом видео,
# а не запись человека отсутствует в регистрационной галерее.
absent_portrait = cv2.imread(str(PHOTOS/'queries/biden.jpg'))
absent_crop = lab.align_face(absent_portrait,lab.largest_face(absent_portrait))
absent_vector = lab.encode_face(absent_portrait)
absent_ranked = sorted(hamilton_tracks,key=lambda t:float(t['vector']@absent_vector),reverse=True)
absent_hits = []
for track in absent_ranked[:3]:
    best_frame = max(track['records'],key=lambda r:float(r['vector']@absent_vector))
    absent_hits.append({'image':best_frame['crop'],'label':f'track {track["track"]}',
                        'score':float(track['vector']@absent_vector),'file':'hamilton.mp4','time_s':best_frame['time']})
absent_video_result = lab.show_search_results(absent_crop,absent_hits,'Query отсутствует в видео: ближайшие всё равно есть')
print('Лучший score отсутствующего query:',round(absent_hits[0]['score'],3),
      '| демонстрационное решение:', 'candidate' if absent_hits[0]['score']>=MATCH_THRESHOLD else 'unknown')
lab.play_video(VIDEOS/'hamilton.mp4')

In [ ]:
clip_records = {}
query_clip = '_a-pB_5eRt0_6'
for path in sorted((VIDEOS/'celebv_hq').glob('*.mp4')):
    analysis = engine.analyze_faces(path,duration=None,sample_fps=4,embeddings=True)
    # Портретные клипы: группируем все пригодные наблюдения одного файла.
    clip_records[path.stem] = analysis['records']
clip_query_crop = clip_records[query_clip][0]['crop']
clip_query = lab.arcface_crop(clip_query_crop)
cv2.imwrite(str(RESULTS/'clip_query.jpg'),clip_query_crop)
clip_ranking = []
for name,records in clip_records.items():
    if name==query_clip:continue
    selected = sorted(records,key=lambda r:r['quality'],reverse=True)[:5]
    vector = lab.unit(np.mean([r['vector'] for r in selected],axis=0))
    best = max(records,key=lambda r:float(r['vector']@clip_query))
    clip_ranking.append({'image':best['crop'],'label':'клип','score':float(vector@clip_query),
                         'file':name+'.mp4','time_s':best['time']})
clip_ranking = sorted(clip_ranking,key=lambda r:r['score'],reverse=True)
clip_result = lab.show_search_results(clip_query_crop,clip_ranking[:3], 'В каком файле встречается query?')
lab.play_video(VIDEOS/'celebv_hq'/clip_ranking[0]['file'],width=480)

---
## 7. Автоматически собираем «состав персонажей» без готовой галереи

До сих пор у нас был референс или запись A/B. Теперь вход — **только видео**. Никакие фотографии из `enrollment`, имена, `person_map` и заранее заданное число людей не передаются алгоритму группировки.

Соберём наблюдения из непрерывного ролика с двумя людьми, прогулки по коридору и Hamilton. Сначала разобьём длинные наблюдения на короткие **треклеты по пять секунд**.

В каждом окне оставим несколько хороших кадров и усредним эмбеддинги. Мелкие лица и окна с недостаточным числом пригодных наблюдений исключаем явно. Затем группируем близкие векторы: «эти короткие наблюдения, возможно, относятся к одному персонажу».

In [ ]:
# Используем только наблюдения и эмбеддинги; walking_person_map не читается.
discovery_inputs = [face_analysis, walking_analysis, hamilton_analysis]
raw_tracklets = engine.make_tracklets(discovery_inputs,seconds=5,min_observations=3)
# Нужны три пригодных кадра, а не один удачный среди двух испорченных.
tracklets = engine.make_tracklets(discovery_inputs,seconds=5,min_observations=3,require_quality=True)
assert len(tracklets)>=2
discovery_vectors = np.stack([tracklet['vector'] for tracklet in tracklets])
print('Треклетов до фильтра:',len(raw_tracklets),'| участвуют в группировке:',len(tracklets))
print('Передаваемый вход кластеризации:',discovery_vectors.shape,'— только векторы, без person ID.')

### Как объединять группы и не смешать цепочку разных лиц

Начинаем с отдельной группы для каждого треклета. Сближаем группы, если **все попарные расстояния между ними** достаточно малы.


In [ ]:
cluster_sensitivity = []
for distance_threshold in [.35,.45,.50,.55]:
    trial_labels = engine.complete_link_clusters(discovery_vectors,max_distance=distance_threshold)
    cluster_sensitivity.append({'max_distance':distance_threshold,
                                'получено групп':len(np.unique(trial_labels))})
display(pd.DataFrame(cluster_sensitivity))

# Выбранная настройка фиксирована; никакого списка зарегистрированных людей здесь нет.
cast_labels = engine.complete_link_clusters(discovery_vectors,max_distance=.50)
cast_ids = sorted(np.unique(cast_labels))
print('Получены анонимные группы:',len(cast_ids))
sorted_indices = np.argsort(cast_labels)
matrix = discovery_vectors[sorted_indices] @ discovery_vectors[sorted_indices].T
plt.figure(figsize=(7,6));plt.imshow(matrix,vmin=0,vmax=1,cmap='viridis');plt.colorbar(label='cosine')
plt.xlabel('треклеты, упорядоченные по группе');plt.ylabel('треклеты')
plt.title('Совпадения и различия без галереи');plt.tight_layout();plt.show()

### Галерея найденных персонажей и время появления

Для каждой группы показываем **несколько** фотографий из разных окон. Это позволяет заметить две типичные ошибки: один человек разделился на несколько групп либо в одной группе смешались разные люди.


In [ ]:
cast_interval_rows = []
cast_group_vectors = []
for group_id in cast_ids:
    members = [tracklets[i] for i in range(len(tracklets)) if cast_labels[i]==group_id]
    representatives = sorted([t['representative'] for t in members],key=lambda r:r['quality'],reverse=True)
    # Сначала показываем разные источники: иначе три самых резких кадра
    # одного файла скроют, что группа объединила персонажа между видео.
    shown = []
    shown_sources = set()
    for representative in representatives:
        if representative['source'] not in shown_sources:
            shown.append(representative); shown_sources.add(representative['source'])
        if len(shown)==3: break
    for representative in representatives:
        if len(shown)==3: break
        if not any(representative is existing for existing in shown): shown.append(representative)
    lab.show_grid([r['crop'] for r in shown],
                  [f'cast_{group_id:02d} | {r["source"]} | {r["time"]:.2f}s' for r in shown],cols=3,size=3)
    cast_group_vectors.append(lab.unit(np.mean([m['vector'] for m in members],axis=0)))
    sources = sorted(set(r['source'] for member in members for r in member['records']))
    for source in sources:
        times = [r['time'] for member in members for r in member['records'] if r['source']==source]
        for start,end in engine.intervals(times,max_gap=1):
            cast_interval_rows.append({'cast_id':f'cast_{group_id:02d}','file':source,
                                       'start_s':start,'end_s':end,'tracklets':len(members)})
cast_intervals = pd.DataFrame(cast_interval_rows)
display(cast_intervals.round(2))
cast_intervals.to_csv(RESULTS/'cast_appearances.csv',index=False)
np.savez_compressed(RESULTS/'cast_index.npz',vectors=discovery_vectors,cluster=cast_labels)

### Проверяем границу между группами


In [ ]:
if len(cast_ids)>1:
    group_vectors = np.stack(cast_group_vectors)
    group_similarity = group_vectors @ group_vectors.T
    np.fill_diagonal(group_similarity,-np.inf)
    i,j = np.unravel_index(np.argmax(group_similarity),group_similarity.shape)
    nearby_groups = []
    nearby_titles = []
    for group_id in [cast_ids[i],cast_ids[j]]:
        members = [tracklets[k] for k in range(len(tracklets)) if cast_labels[k]==group_id]
        best = max((m['representative'] for m in members),key=lambda r:r['quality'])
        nearby_groups.append(best['crop']);nearby_titles.append(f'cast_{group_id:02d}')
    lab.show_grid(nearby_groups,nearby_titles,cols=2,size=3)
    print('Ближайшие разные группы: cosine прототипов =',round(float(group_similarity[i,j]),3))

# Итоговый ролик показывает группы на реальных кадрах прогулки.
key_to_cast = {tracklets[i]['key']:f'cast_{int(cast_labels[i]):02d}' for i in range(len(tracklets))}
by_frame = {}
for r in walking_analysis['records']:
    by_frame.setdefault(r['frame_index'],[]).append(r)
cast_video = []
for frame_index,frame in enumerate(walking_analysis['frames']):
    out = frame.copy()
    for r in by_frame.get(frame_index,[]):
        key = (r['source'],r['track'],int(r['time']//5))
        label = key_to_cast.get(key,'unassigned')
        x,y,w,h = r['box'].astype(int)
        cv2.rectangle(out,(x,y),(x+w,y+h),(0,220,0),2)
        cv2.putText(out,label,(max(0,x),max(18,y-6)),cv2.FONT_HERSHEY_SIMPLEX,.55,(0,220,0),2)
    cast_video.append(out)
lab.save_video(cast_video,RESULTS/'videos/discovered_cast.mp4',walking_analysis['fps'])
print('Сохранены cast_index.npz, cast_appearances.csv и видео без заранее заданных имён.')